## 6. Macro Credit Risk — Vintage Analysis

Vintage analysis tracks cohorts of loans by origination period and compares their default trajectories.

**Why it matters:** If a new vintage's cumulative default curve is rising faster than previous vintages at the same loan age — that's an early warning signal. IFRS 9 uses this logic for lifetime PD estimation.

In [ ]:
df_cleaned['issue_d'] = pd.to_datetime(df_cleaned['issue_d'])
df_cleaned['issue_year'] = df_cleaned['issue_d'].dt.year
df_cleaned['issue_quarter'] = df_cleaned['issue_d'].dt.to_period('Q')

fig, axes = plt.subplots(2, 1, figsize=(15, 14))

# Part 1: Quarterly macro default rate
quarterly_data = df_cleaned.groupby('issue_quarter')['default'].mean() * 100
sns.lineplot(x=quarterly_data.index.to_timestamp(), y=quarterly_data.values,
             ax=axes[0], color='#2c3e50', linewidth=2.5, marker='o')
axes[0].set_title('Historical Macro Default Rate by Origination Quarter', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Default Rate (%)')
axes[0].grid(True, linestyle='--', alpha=0.6)

# Part 2: Cumulative lifecycle curves per vintage
if 'last_pymnt_d' in df_cleaned.columns:
    df_cleaned['last_pymnt_d'] = pd.to_datetime(df_cleaned['last_pymnt_d'])
    df_cleaned['months_since_issue'] = (
        (df_cleaned['last_pymnt_d'].dt.year - df_cleaned['issue_d'].dt.year) * 12 +
        (df_cleaned['last_pymnt_d'].dt.month - df_cleaned['issue_d'].dt.month)
    ).clip(0, 36)
else:
    df_cleaned['months_since_issue'] = df_cleaned['grade'].map(
        {'A':5, 'B':10, 'C':15, 'D':20, 'E':25, 'F':30, 'G':36})

vintage_years = sorted([y for y in df_cleaned['issue_year'].unique() if 2011 <= y <= 2018])
colors = sns.color_palette("viridis", len(vintage_years))
months_axis = sorted(df_cleaned['months_since_issue'].dropna().unique())

for idx, year in enumerate(vintage_years):
    cohort = df_cleaned[df_cleaned['issue_year'] == year]
    total = len(cohort)
    if total > 5000:
        running = 0
        rates = []
        for m in months_axis:
            running += len(cohort[(cohort['months_since_issue'] == m) & (cohort['default'] == 1)])
            rates.append(running / total * 100)
        axes[1].plot(months_axis, rates, marker='x', label=f'{year}', color=colors[idx], linewidth=2)

axes[1].set_title('Cumulative Default Curves by Origination Vintage', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Months Elapsed Since Origination')
axes[1].set_ylabel('Cumulative Default Rate (%)')
axes[1].legend(title='Vintage Year', bbox_to_anchor=(1.02, 1), loc='upper left')
axes[1].grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()